### Notebook to train the Teacher and students models

In [1]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('githubAccess')
os.environ['WANDB_API_KEY'] = userdata.get('wandbKey')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('anthropic_key')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

%cd /content/drive/MyDrive/incremental_learning_on_the_edge/src/core

/content/drive/MyDrive/incremental_learning_on_the_edge/src/core


In [3]:
!git pull

remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 10 (delta 6), reused 9 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 2.46 KiB | 1024 bytes/s, done.
From https://github.com/BrunoSantome/incremental_learning_on_the_edge
   1ffc360..3d56bde  main       -> origin/main
Updating 1ffc360..3d56bde
Fast-forward
 src/core/distillation_1.py | 133 +++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 133 insertions(+)


In [4]:
%pip install -q "transformers>=4.48,<5" "huggingface_hub>=0.34,<1" "datasets<3.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 15.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have h

In [9]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.1 MB/s eta 0:00:00


In [ ]:
#If errors, run this and restart runTime
#!pip install -q numpy==1.26.4 datasets==2.14.7 huggingface_hub==0.17.3 transformers==4.35.0

In [4]:
import sys
import wandb
import torch
import pandas as pd
from dotenv import load_dotenv
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)
sys.path.append("..")
from core.configuration import load_config
from core.dataloader import DataClass
from core.utils import build_model
from core.train import Trainer
from core.distillation_1 import (
    DistillationTrainer,
    DistillationTrainerV0,
    run_distillation_training,
    run_incremental_step,
    run_incremental_experiment,
    run_synthetic_incremental_experiment,
    run_production_incremental_experiment,
    run_production_step
    )


In [5]:
import transformers, sys
print(transformers.__version__)
print(transformers.__file__)
print(sys.executable)

4.57.6
/usr/local/lib/python3.13/dist-packages/transformers/__init__.py
/usr/bin/python3


In [ ]:
# def run_distillation_training(
#     dataclass, teacher_model, teacher_tokenizer, models_keys, config
# ):
#     for key in models_keys:
#         student_model, student_tokenizer, _ = build_model(config[key]["name"], dataclass.num_labels)
#         dft_pre_dataloader = dataclass.get_dataloader_data(key, student_tokenizer, keep_utt=True)
#         trainer = DistillationTrainer(
#             student_model=student_model,
#             teacher_model=teacher_model,
#             student_name=key,
#             teacher_tokenizer=teacher_tokenizer,
#             student_dataloaders=dft_pre_dataloader,
#             config=config,
#         )
#         trainer.train()

### Running the distillation loop over the V0 in EN and Default hyperparameters. For tuning change config.yaml

In [ ]:
config = load_config()
dataclass = DataClass()
print("Dataset Loaded")
teacher_path = "outputs/checkpoints/teacher_model"
teacher_model = AutoModelForSequenceClassification.from_pretrained(teacher_path)
teacher_tokenizer = AutoTokenizer.from_pretrained(config["teacher"]["name"])
model_keys = ["student1"]
run_distillation_training(dataclass, teacher_model, teacher_tokenizer, model_keys, config)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The repository for qanastek/MASSIVE contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/qanastek/MASSIVE.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

...

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at jhu-clsp/ettin-encoder-68m and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/3622 [00:00<?, ? examples/s]

Map:   0%|          | 0/914 [00:00<?, ? examples/s]

Map:   0%|          | 0/657 [00:00<?, ? examples/s]

ValueError: Label-space mismatch: student=20, teacher=15. Distillation requires matching classifier heads.

### Running test of incremental distillation re-training loop in EN and Default hyperparameters. For tuning change config.yaml.


#### We use V0 as teacher and increment the model knowledge by a single intent into the V1 version.

In [ ]:
# config = load_config()
# dataclass = DataClass()

# print("Dataset Loaded")
# print("num_labels:", dataclass.num_labels)                       # must be 15  ==> V0
# print("reserve intents:", dataclass.dataset_totrain["train_set"].unique("intent"))
# new_intent_name = "general_joke"
# run_incremental_step(
#     dataclass=dataclass,
#     new_intent_name=new_intent_name,
#     student_key="student1",
#     config=config,
#     version=1,      # creates version V1 of incremental model
#     K=50,
# )

### Run the experimental run, up to 5 new intents added, so 5 incremental steps

In [6]:
config = load_config()
dataclass = DataClass()
print("Dataset Loaded")
print("num_labels:", dataclass.num_labels)
print("reserve intents:", dataclass.dataset_totrain["train_set"].unique("intent"))
#reserve intents: ['takeaway_order', 'general_joke', 'recommendation_locations', 'play_podcasts', 'transport_traffic']

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The repository for qanastek/MASSIVE contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/qanastek/MASSIVE.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

...

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/690 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/122 [00:00<?, ? examples/s]

Dataset Loaded
num_labels: 20
reserve intents: ['takeaway_order', 'general_joke', 'recommendation_locations', 'play_podcasts', 'transport_traffic']


In [ ]:
# intents_to_add = ["general_joke"]
# run_incremental_experiment(intents_to_add, "student1", config, 60, 42)


Now running 5 incremental steps

In [ ]:
intents_to_add = ['takeaway_order', 'general_joke', 'recommendation_locations', 'play_podcasts', 'transport_traffic']
run_incremental_experiment(intents_to_add, "student1", config, 70, 42)

v1: adding intent: takeaway_order


Filter:   0%|          | 0/690 [00:00<?, ? examples/s]

Filter:   0%|          | 0/690 [00:00<?, ? examples/s]

Map:   0%|          | 0/135 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/135 [00:00<?, ? examples/s]

...

testing epoch: 2/20: 100%|██████████| 25/25 [00:00<00:00, 31.24it/s]


Saving checkpoint


testing epoch: 7/20: 100%|██████████| 25/25 [00:00<00:00, 30.55it/s]


Early stopping at epoch 7


epoch,▁▂▃▄▅▆▇█
eval_accuracy,▇▂█▁▆▆▅▇
eval_accuracy_en-US,▇▂█▁▆▆▅▇
eval_f1_alarm_set,████▁███
eval_f1_audio_volume_down,▁▁▁▁▁▁▁▁
eval_f1_audio_volume_mute,▁▁▁▁█▁▁▁
eval_f1_audio_volume_up,██▁██▁▁▁
eval_f1_datetime_query,█▁█▁▁███
eval_f1_email_addcontact,▁██▁▁▁▁▁
eval_f1_general_joke,██▄▁▄█▄▄
+21,...


In [ ]:
# !find . -name "*.safetensors" -o -name "pytorch_model.bin" 2>/dev/null

### Run the incremental step loop with synthetic LLM generated data
Using synthetic train real eval and test data

In [ ]:
config = load_config()
dataclass = DataClass()
print("Dataset Loaded")
print("num_labels:", dataclass.num_labels)
print("reserve intents:", dataclass.dataset_totrain["train_set"].unique("intent"))
intents_to_add = ['takeaway_order', 'general_joke', 'recommendation_locations', 'play_podcasts', 'transport_traffic']
dataclass = run_synthetic_incremental_experiment(
        intents_to_add=intents_to_add,
        student_key="student1",
        config=config,
        K=70, #replay-buffer exemplars per intent
        n_generate=70, # synthetic utterances to request per new intent, matching the K number to avoid under-represented new classes in the balanced replay buffer
    )


### Saving synthetic dataset on disk

In [ ]:
# Save dataset with synthetic examples
dataclass.dataset_pretraining.save_to_disk("src/outputs/datasets/SYNTH_K70_g70_a0.2_s42")

Saving the dataset (0/1 shards):   0%|          | 0/4312 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1064 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/779 [00:00<?, ? examples/s]

In [ ]:
# from datasets import load_from_disk
# ds = load_from_disk("src/outputs/datasets/SYNTH_K70_g70_a0.2_s42")
# synth = ds["train_set"].filter(lambda ex: ex["id"].startswith("synth_"))
# print(synth["utt"])

### Production Experiment synthetic train/eval/test, reserved intents

Everything is synthetic but we use the same 5 reserved intents, not new ones
in order to compare real train/eval/test pipeline with the same intents

In [7]:
intents_to_add = ['takeaway_order', 'general_joke', 'recommendation_locations', 'play_podcasts', 'transport_traffic']
dataclass = run_production_incremental_experiment(
    intents_to_add=intents_to_add,
    student_key="student1",
    config=config,
    K=70,
    n_generate=130, # more than the split just in case it generated duplicates
    n_eval=20,
    n_test=20,
    seed=42,
)

v1: generating all-synthetic intent: takeaway_order


Casting the dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

testing epoch: 2/20: 100%|██████████| 24/24 [00:00<00:00, 32.89it/s]


Saving checkpoint


testing epoch: 7/20: 100%|██████████| 24/24 [00:00<00:00, 33.32it/s]


Early stopping at epoch 7


epoch,▁▂▃▄▅▆▇█
eval_accuracy,▁▆▅▆█▆▆█
eval_accuracy_en-US,▁▆▅▆█▆▆█
eval_f1_alarm_set,█▁███▁▁▁
eval_f1_audio_volume_down,▁▁▁▁▁▁▁▁
eval_f1_audio_volume_mute,██▁█████
eval_f1_audio_volume_up,▁▁█▁▁▁▁▁
eval_f1_datetime_query,█▁██████
eval_f1_email_addcontact,▁▁▁▁▁▁▁▁
eval_f1_general_joke,█▁█▁▁▁▁▁
+21,...


### Evaluating double test set synthetic vs real

In [12]:
from core.evaluate import  intents_report, old_intent_persistance_table, new_intent_acquisition_table

In [9]:
rows_syn,  m_syn  = intents_report(dataclass, "student1", config, n_versions=5, test_source="pretraining")
rows_real, m_real = intents_report(dataclass, "student1", config, n_versions=5, test_source="real")

Filter:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/914 [00:00<?, ? examples/s]

evaluating test set: 100%|██████████| 29/29 [00:00<00:00, 33.35it/s]


V0: has 15 intents on the test split


Filter:   0%|          | 0/1014 [00:00<?, ? examples/s]

...

V4: has 19 intents on the test split


Filter:   0%|          | 0/1064 [00:00<?, ? examples/s]

Map:   0%|          | 0/1064 [00:00<?, ? examples/s]

evaluating test set: 100%|██████████| 34/34 [00:01<00:00, 33.70it/s]


V5: has 20 intents on the test split


### Evaluation of old intents

In [10]:
print(old_intent_persistance_table(rows_syn, dataclass.id2intent))

                         V0        V1        V2        V3        V4        V5
alarm_set          0.975610  0.975610  0.962963  0.950000  0.962963  0.950000
audio_volume_down  1.000000  0.952381  1.000000  1.000000  1.000000  1.000000
audio_volume_mute  0.939394  0.939394  0.968750  0.953846  0.939394  0.953846
audio_volume_up    0.916667  0.880000  0.916667  0.916667  0.916667  0.869565
datetime_query     0.960452  0.954545  0.960000  0.954545  0.960000  0.965517
email_addcontact   0.923077  0.923077  0.960000  0.960000  0.960000  0.960000
lists_createoradd  0.945946  0.918919  0.935065  0.935065  0.935065  0.947368
news_query         0.964427  0.976000  0.971429  0.967213  0.966942  0.953975
play_audiobook     0.805195  0.815789  0.790123  0.800000  0.800000  0.800000
play_game          0.861538  0.875000  0.857143  0.845070  0.833333  0.857143
play_music         0.928962  0.933702  0.906433  0.902655  0.910714  0.897590
play_radio         0.936170  0.936170  0.937063  0.944444  0.944

### evaluation on synthetic test set

In [12]:
print(new_intent_acquisition_table(rows_syn, dataclass.id2intent))

                            V0        V1        V2        V3        V4  \
takeaway_order            None  0.784314  0.769231  0.833333  0.869565   
general_joke              None       NaN  0.754717  0.689655  0.769231   
recommendation_locations  None       NaN       NaN  0.769231  0.851064   
play_podcasts             None       NaN       NaN       NaN  0.816327   
transport_traffic         None       NaN       NaN       NaN       NaN   
mean_new                   NaN  0.784314  0.761974  0.764073  0.826547   

                                V5  
takeaway_order            0.816327  
general_joke              0.800000  
recommendation_locations  0.851064  
play_podcasts             0.769231  
transport_traffic         0.730769  
mean_new                  0.793478  


### evaluation on real test set

In [11]:
print(new_intent_acquisition_table(rows_real, dataclass.id2intent))

                            V0        V1        V2        V3        V4  \
takeaway_order            None  0.754717  0.763636  0.800000  0.808511   
general_joke              None       NaN  0.745098  0.678571  0.760000   
recommendation_locations  None       NaN       NaN  0.800000  0.840580   
play_podcasts             None       NaN       NaN       NaN  0.859375   
transport_traffic         None       NaN       NaN       NaN       NaN   
mean_new                   NaN  0.754717  0.754367  0.759524  0.817116   

                                V5  
takeaway_order            0.760000  
general_joke              0.791667  
recommendation_locations  0.840580  
play_podcasts             0.839695  
transport_traffic         0.651163  
mean_new                  0.776621  


### Production incremental step, synthetic train/eval/test, new intents

In [7]:
config = load_config()
# resetting old intent registry_path to 15 again.
reg = DataClass._resolve_path(config["registry_path"])
if os.path.exists(reg):
    os.remove(reg)
dataclass = DataClass()

Filter:   0%|          | 0/11514 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2974 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2033 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11514 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2974 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/690 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/122 [00:00<?, ? examples/s]

In [8]:
escalated_utts = ['could you open the window of the front of the car']

In [11]:
intent_name, output_dir = run_production_step(
    dataclass=dataclass,
    escalated_utts=escalated_utts,
    student_key="student1",
    config=config,
    version=1, #only produce 1 new intent on this run
    K=70,
    n_generate=130, # more than the split just in case it generated duplicates
    n_eval=20,
    n_test=20,
    seed=42,
)
print("assigned name:", intent_name)
print("output dir   :", output_dir)

v1: production intent 'car_window_control'


Casting the dataset:   0%|          | 0/93 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Saving checkpoint


testing epoch: 16/20: 100%|██████████| 22/22 [00:00<00:00, 32.48it/s]


Early stopping at epoch 16


epoch,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
eval_accuracy,▇▆▁▇▇█▆▅▇▇▇▇▇▇▇▇▇
eval_accuracy_en-US,▇▆▁▇▇█▆▅▇▇▇▇▇▇▇▇▇
eval_f1_alarm_set,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval_f1_audio_volume_down,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval_f1_audio_volume_mute,███▁█████████████
eval_f1_audio_volume_up,▁█▁▁███▁▁▁▁▁▁▁▁▁▁
eval_f1_car_window_control,▅▄▁▂▅▆▆█▇▇███████
eval_f1_datetime_query,▁██████▁██▁▁▁▁▁▁▁
eval_f1_email_addcontact,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+17,...


assigned name: car_window_control
output dir   : outputs/checkpoints/student_ettin_68m_distill_v1


In [13]:
from core.evaluate import  intents_report, old_intent_persistance_table, new_intent_acquisition_table

rows, metrics = intents_report(
    dataclass, "student1", config,
    n_versions=1, # only ran 1 incremental step
    test_source="pretraining",
)

print(old_intent_persistance_table(rows, dataclass.id2intent))
print(new_intent_acquisition_table(rows, dataclass.id2intent))

Filter:   0%|          | 0/934 [00:00<?, ? examples/s]

Map:   0%|          | 0/914 [00:00<?, ? examples/s]

evaluating test set: 100%|██████████| 29/29 [00:00<00:00, 32.67it/s]


V0: has 15 intents on the test split


Filter:   0%|          | 0/934 [00:00<?, ? examples/s]

Map:   0%|          | 0/934 [00:00<?, ? examples/s]

evaluating test set: 100%|██████████| 30/30 [00:00<00:00, 32.52it/s]


V1: has 16 intents on the test split
                         V0        V1
alarm_set          0.975610  0.975610
audio_volume_down  1.000000  0.952381
audio_volume_mute  0.939394  0.953846
audio_volume_up    0.916667  0.800000
datetime_query     0.960452  0.955056
email_addcontact   0.923077  0.923077
lists_createoradd  0.945946  0.947368
news_query         0.964427  0.968000
play_audiobook     0.805195  0.800000
play_game          0.861538  0.882353
play_music         0.928962  0.937500
play_radio         0.936170  0.930556
transport_query    0.907216  0.905263
transport_taxi     1.000000  1.000000
weather_query      0.980769  0.980645
mean_old           0.936362  0.927444
                      V0   V1
car_window_control  None  0.8
mean_new             NaN  0.8


In [14]:
new_idx = dataclass.registry[intent_name]
for split in dataclass.sets_names:          # train_set, test_set, eval_set
      ds = dataclass.dataset_pretraining[split]
      rows = ds.filter(lambda ex: ex[dataclass.label_col] == new_idx)
      print(f"=== {split}: {len(rows)} rows ===")
      for r in rows:
          print(f"  [{r['id']}] {r['utt']}")
      print()

Filter:   0%|          | 0/3715 [00:00<?, ? examples/s]

=== train_set: 93 rows ===
  [synth_car_window_control_train_0] roll down the rear driver side window
  [synth_car_window_control_train_1] lower the window a couple inches
  [synth_car_window_control_train_2] lower the glass on my door
  [synth_car_window_control_train_3] shut the window it's windy
  [synth_car_window_control_train_4] seal up all the windows
  [synth_car_window_control_train_5] close the passenger windows but leave mine open
  [synth_car_window_control_train_6] roll up the window behind the driver
  [synth_car_window_control_train_7] put the right side windows down
  [synth_car_window_control_train_8] close all the car windows now
  [synth_car_window_control_train_9] close the window in the back
  [synth_car_window_control_train_10] take all windows down
  [synth_car_window_control_train_11] put the window down for the parking ticket
  [synth_car_window_control_train_12] let the rear glass down
  [synth_car_window_control_train_13] raise the passenger window a little
 

Filter:   0%|          | 0/934 [00:00<?, ? examples/s]

=== test_set: 20 rows ===
  [synth_car_window_control_test_0] open the window by the driver
  [synth_car_window_control_test_1] let the passenger window down
  [synth_car_window_control_test_2] can you close the back left window
  [synth_car_window_control_test_3] open the back seat windows
  [synth_car_window_control_test_4] open the driver window about halfway
  [synth_car_window_control_test_5] windows up
  [synth_car_window_control_test_6] roll down the driver side window
  [synth_car_window_control_test_7] hey can you open my window
  [synth_car_window_control_test_8] let some air in open the windows
  [synth_car_window_control_test_9] raise the front passenger glass
  [synth_car_window_control_test_10] put the rear windows down
  [synth_car_window_control_test_11] close the rear left window
  [synth_car_window_control_test_12] open just my window please
  [synth_car_window_control_test_13] crack the driver window for the dog
  [synth_car_window_control_test_14] close everything u

Filter:   0%|          | 0/677 [00:00<?, ? examples/s]

=== eval_set: 20 rows ===
  [synth_car_window_control_eval_0] i need the window open on the passenger side
  [synth_car_window_control_eval_1] lower the passenger side glass all the way
  [synth_car_window_control_eval_2] put my window up please
  [synth_car_window_control_eval_3] it's getting cold close my window
  [synth_car_window_control_eval_4] open the front two windows
  [synth_car_window_control_eval_5] can you shut the window next to the kid
  [synth_car_window_control_eval_6] raise the rear right window
  [synth_car_window_control_eval_7] roll down the back right window
  [synth_car_window_control_eval_8] close the driver's window all the way
  [synth_car_window_control_eval_9] open the front windows please
  [synth_car_window_control_eval_10] open the window for the drive through
  [synth_car_window_control_eval_11] please raise the window next to the child
  [synth_car_window_control_eval_12] can you shut the rear windows
  [synth_car_window_control_eval_13] please open all

### Evaluating the models



In [ ]:

config = load_config()
dataclass = DataClass()
intents = ['takeaway_order', 'general_joke', 'recommendation_locations', 'play_podcasts', 'transport_traffic']
for name in intents:
  dataclass.admit_intent(name)

rows, metrics = intents_report(dataclass, "student1", config, n_versions=len(intents))
print(old_intent_persistance_table(rows, dataclass.id2intent))
print(new_intent_acquisition_table(rows, dataclass.id2intent))

Filter:   0%|          | 0/1064 [00:00<?, ? examples/s]

Map:   0%|          | 0/914 [00:00<?, ? examples/s]

evaluating test set: 100%|██████████| 29/29 [00:00<00:00, 30.25it/s]


V0: has 15 intents on the test split


Filter:   0%|          | 0/1064 [00:00<?, ? examples/s]

...

V4: has 19 intents on the test split


Filter:   0%|          | 0/1064 [00:00<?, ? examples/s]

Map:   0%|          | 0/1064 [00:00<?, ? examples/s]

evaluating test set: 100%|██████████| 34/34 [00:01<00:00, 29.51it/s]


V5: has 20 intents on the test split
                         V0        V1        V2        V3        V4        V5
alarm_set          0.975610  0.975610  0.963855  0.963855  0.975610  0.975610
audio_volume_down  1.000000  1.000000  1.000000  0.952381  1.000000  0.952381
audio_volume_mute  0.939394  0.939394  0.939394  0.939394  0.937500  0.937500
audio_volume_up    0.916667  0.869565  0.869565  0.923077  0.833333  0.846154
datetime_query     0.960452  0.960000  0.965909  0.960000  0.971751  0.959538
email_addcontact   0.923077  0.923077  0.923077  0.923077  0.923077  0.923077
lists_createoradd  0.945946  0.918919  0.918919  0.933333  0.933333  0.909091
news_query         0.964427  0.979757  0.967213  0.962656  0.949580  0.917031
play_audiobook     0.805195  0.784810  0.756098  0.775000  0.794521  0.760563
play_game          0.861538  0.840580  0.852941  0.882353  0.819672  0.857143
play_music         0.928962  0.935574  0.906977  0.910145  0.905556  0.904494
play_radio         0.936170

In [ ]:
"""

                         V0        V1        V2        V3        V4        V5
alarm_set          0.975610  0.975610  0.963855  0.963855  0.975610  0.975610
audio_volume_down  1.000000  1.000000  1.000000  0.952381  1.000000  0.952381
audio_volume_mute  0.939394  0.939394  0.939394  0.939394  0.937500  0.937500
audio_volume_up    0.916667  0.869565  0.869565  0.923077  0.833333  0.846154
datetime_query     0.960452  0.960000  0.965909  0.960000  0.971751  0.959538
email_addcontact   0.923077  0.923077  0.923077  0.923077  0.923077  0.923077
lists_createoradd  0.945946  0.918919  0.918919  0.933333  0.933333  0.909091
news_query         0.964427  0.979757  0.967213  0.962656  0.949580  0.917031
play_audiobook     0.805195  0.784810  0.756098  0.775000  0.794521  0.760563
play_game          0.861538  0.840580  0.852941  0.882353  0.819672  0.857143
play_music         0.928962  0.935574  0.906977  0.910145  0.905556  0.904494
play_radio         0.936170  0.936170  0.937063  0.937063  0.921986  0.928571
transport_query    0.907216  0.938776  0.929293  0.926316  0.929293  0.927835
transport_taxi     1.000000  1.000000  1.000000  1.000000  1.000000  1.000000
weather_query      0.980769  0.984127  0.974194  0.983819  0.983713  0.980519
mean_old           0.936362  0.932424  0.926967  0.931498  0.925262  0.918634
                            V0        V1        V2        V3        V4  \
takeaway_order            None  0.807692  0.733333  0.862745  0.833333
general_joke              None       NaN  0.863636  0.883721  0.926829
recommendation_locations  None       NaN       NaN  0.784810  0.861111
play_podcasts             None       NaN       NaN       NaN  0.816901
transport_traffic         None       NaN       NaN       NaN       NaN
mean_new                   NaN  0.807692  0.798485  0.843759  0.859544

                                V5
takeaway_order            0.857143
general_joke              0.950000
recommendation_locations  0.939394
play_podcasts             0.842857
transport_traffic         0.555556
mean_new                  0.828990
"""

### Synthetic data evaluation
can synthetic data substitute for real?

In [ ]:
#
config = load_config()
dataclass = DataClass()
intents = ['takeaway_order', 'general_joke', 'recommendation_locations', 'play_podcasts', 'transport_traffic']
for name in intents:
  dataclass.admit_intent(name)

rows, metrics = intents_report(dataclass, "student1", config, n_versions=len(intents))
print(old_intent_persistance_table(rows, dataclass.id2intent))
print(new_intent_acquisition_table(rows, dataclass.id2intent))

evaluating test set: 100%|██████████| 29/29 [00:00<00:00, 31.84it/s]


V0: has 15 intents on the test split


evaluating test set: 100%|██████████| 30/30 [00:00<00:00, 31.66it/s]


V1: has 16 intents on the test split


evaluating test set: 100%|██████████| 30/30 [00:00<00:00, 31.93it/s]


...

V3: has 18 intents on the test split


evaluating test set: 100%|██████████| 33/33 [00:01<00:00, 32.35it/s]


V4: has 19 intents on the test split


evaluating test set: 100%|██████████| 34/34 [00:01<00:00, 32.83it/s]


V5: has 20 intents on the test split
                         V0        V1        V2        V3        V4        V5
alarm_set          0.975610  0.975610  0.963855  0.987654  0.987654  0.975610
audio_volume_down  1.000000  1.000000  0.952381  0.952381  0.952381  1.000000
audio_volume_mute  0.939394  0.939394  0.923077  0.923077  0.923077  0.939394
audio_volume_up    0.916667  0.869565  0.846154  0.800000  0.846154  0.916667
datetime_query     0.960452  0.960000  0.965909  0.971429  0.971429  0.971098
email_addcontact   0.923077  0.923077  0.923077  0.960000  0.923077  0.923077
lists_createoradd  0.945946  0.933333  0.921053  0.947368  0.909091  0.897436
news_query         0.964427  0.975806  0.954357  0.945148  0.953975  0.953975
play_audiobook     0.805195  0.805195  0.790123  0.790123  0.771084  0.765432
play_game          0.861538  0.869565  0.878788  0.869565  0.845070  0.845070
play_music         0.928962  0.938202  0.917847  0.919075  0.913295  0.915942
play_radio         0.936170

In [ ]:
"""
Run 1/3 of synthetic incremental loop
                        V0        V1        V2        V3        V4        V5
alarm_set          0.975610  0.975610  0.987654  0.987654  0.975610  0.987654
audio_volume_down  1.000000  1.000000  1.000000  1.000000  0.952381  1.000000
audio_volume_mute  0.939394  0.939394  0.969697  0.939394  0.939394  0.939394
audio_volume_up    0.916667  0.869565  0.869565  0.916667  0.880000  0.916667
datetime_query     0.960452  0.960000  0.966292  0.960894  0.954545  0.971429
email_addcontact   0.923077  0.923077  0.960000  0.960000  0.960000  0.960000
lists_createoradd  0.945946  0.918919  0.945946  0.935065  0.923077  0.911392
news_query         0.964427  0.976000  0.955466  0.959016  0.967213  0.936709
play_audiobook     0.805195  0.815789  0.826667  0.815789  0.805195  0.784810
play_game          0.861538  0.865672  0.878788  0.895522  0.892308  0.869565
play_music         0.928962  0.932961  0.937500  0.929972  0.917847  0.914286
play_radio         0.936170  0.936170  0.951049  0.919708  0.943662  0.944444
transport_query    0.907216  0.918367  0.907216  0.926316  0.926316  0.926316
transport_taxi     1.000000  1.000000  1.000000  0.977778  1.000000  1.000000
weather_query      0.980769  0.983923  0.977346  0.983923  0.977346  0.977199
mean_old           0.936362  0.934363  0.942212  0.940513  0.934326  0.935991
                            V0       V1        V2        V3        V4  \
takeaway_order            None  0.77193  0.897959  0.933333  0.909091
general_joke              None      NaN  0.666667  0.716981  0.791667
recommendation_locations  None      NaN       NaN  0.882353  0.845070
play_podcasts             None      NaN       NaN       NaN  0.854962
transport_traffic         None      NaN       NaN       NaN       NaN
mean_new                   NaN  0.77193  0.782313  0.844222  0.850197

                                V5
takeaway_order            0.909091
general_joke              0.791667
recommendation_locations  0.882353
play_podcasts             0.873016
transport_traffic         0.697674
mean_new                  0.830760

"""
#Run 2/3
"""
                         V0        V1        V2        V3        V4        V5
alarm_set          0.975610  0.975610  0.987654  0.987654  0.987654  0.987654
audio_volume_down  1.000000  1.000000  0.900000  0.900000  1.000000  1.000000
audio_volume_mute  0.939394  0.939394  0.937500  0.939394  0.939394  0.939394
audio_volume_up    0.916667  0.869565  0.769231  0.800000  0.960000  0.916667
datetime_query     0.960452  0.960000  0.966292  0.960452  0.960452  0.971429
email_addcontact   0.923077  0.923077  0.923077  0.960000  0.923077  0.923077
lists_createoradd  0.945946  0.933333  0.921053  0.911392  0.894737  0.909091
news_query         0.964427  0.975806  0.975610  0.971193  0.966942  0.958678
play_audiobook     0.805195  0.805195  0.790123  0.800000  0.790123  0.767442
play_game          0.861538  0.869565  0.869565  0.857143  0.845070  0.857143
play_music         0.928962  0.929178  0.921739  0.934097  0.911175  0.913295
play_radio         0.936170  0.929577  0.930556  0.929577  0.916667  0.930556
transport_query    0.907216  0.927835  0.918367  0.937500  0.927835  0.916667
transport_taxi     1.000000  0.978723  0.978723  0.978723  1.000000  1.000000
weather_query      0.980769  0.977346  0.983819  0.980519  0.980519  0.980645
mean_old           0.936362  0.932947  0.918221  0.923176  0.933576  0.931449
                            V0        V1        V2        V3        V4  \
takeaway_order            None  0.721311  0.846154  0.909091  0.883721
general_joke              None       NaN  0.791667  0.775510  0.826087
recommendation_locations  None       NaN       NaN  0.845070  0.837838
play_podcasts             None       NaN       NaN       NaN  0.887097
transport_traffic         None       NaN       NaN       NaN       NaN
mean_new                   NaN  0.721311  0.818910  0.843224  0.858686

                                V5
takeaway_order            0.904762
general_joke              0.863636
recommendation_locations  0.898551
play_podcasts             0.892562
transport_traffic         0.731707
mean_new                  0.858244

"""
# RUN 3/3
"""
                         V0        V1        V2        V3        V4        V5
alarm_set          0.975610  0.975610  0.963855  0.987654  0.987654  0.975610
audio_volume_down  1.000000  1.000000  0.952381  0.952381  0.952381  1.000000
audio_volume_mute  0.939394  0.939394  0.923077  0.923077  0.923077  0.939394
audio_volume_up    0.916667  0.869565  0.846154  0.800000  0.846154  0.916667
datetime_query     0.960452  0.960000  0.965909  0.971429  0.971429  0.971098
email_addcontact   0.923077  0.923077  0.923077  0.960000  0.923077  0.923077
lists_createoradd  0.945946  0.933333  0.921053  0.947368  0.909091  0.897436
news_query         0.964427  0.975806  0.954357  0.945148  0.953975  0.953975
play_audiobook     0.805195  0.805195  0.790123  0.790123  0.771084  0.765432
play_game          0.861538  0.869565  0.878788  0.869565  0.845070  0.845070
play_music         0.928962  0.938202  0.917847  0.919075  0.913295  0.915942
play_radio         0.936170  0.936170  0.913043  0.929577  0.937063  0.944444
transport_query    0.907216  0.938776  0.929293  0.940000  0.937500  0.926316
transport_taxi     1.000000  1.000000  1.000000  0.978723  1.000000  1.000000
weather_query      0.980769  0.984026  0.980645  0.973856  0.980519  0.980519
mean_old           0.936362  0.936581  0.923974  0.925865  0.923425  0.930332
                            V0   V1        V2        V3        V4        V5
takeaway_order            None  0.8  0.830189  0.823529  0.851064  0.869565
general_joke              None  NaN  0.760000  0.750000  0.745098  0.826087
recommendation_locations  None  NaN       NaN  0.753247  0.810811  0.833333
play_podcasts             None  NaN       NaN       NaN  0.894309  0.887097
transport_traffic         None  NaN       NaN       NaN       NaN  0.750000
mean_new                   NaN  0.8  0.795094  0.775592  0.825320  0.833216
"""

'\nRun 1/3 of synthetic incremental loop\n                         V0        V1        V2        V3        V4        V5\nalarm_set          0.975610  0.975610  0.987654  0.987654  0.975610  0.987654\naudio_volume_down  1.000000  1.000000  1.000000  0.952381  0.952381  0.952381\naudio_volume_mute  0.939394  0.941176  0.941176  0.909091  0.939394  0.923077\naudio_volume_up    0.916667  0.888889  0.923077  0.827586  0.846154  0.769231\ndatetime_query     0.960452  0.955056  0.955056  0.955056  0.960452  0.949721\nemail_addcontact   0.923077  0.923077  0.923077  0.960000  0.960000  0.960000\nlists_createoradd  0.945946  0.931507  0.918919  0.911392  0.923077  0.923077\nnews_query         0.964427  0.983740  0.963265  0.971429  0.966942  0.957983\nplay_audiobook     0.805195  0.765432  0.790123  0.776471  0.780488  0.780488\nplay_game          0.861538  0.852941  0.873239  0.895522  0.882353  0.885714\nplay_music         0.928962  0.931034  0.923529  0.906433  0.906977  0.906977\nplay_radio